# Harmony reference workflow on Dataset 2 (mBDRC renal cortex)

This notebook demonstrates a user-side **Harmony** workflow on registered dataset selector `2` (`mbdrc_renal_cortex`) and then benchmarks only the resulting latent with scRareBench.

Evaluation contract supplied by scRareBench:
- label: `cell_type`;
- evaluation batch: `donor_id x assay` (`scrarebench_batch`);
- fixed registered rare-scenario metadata attached by `load_dataset(2)`;
- scIB reference HVG policy: global Seurat-v3 HVG selection.

The method implementation remains notebook/user code. For mBDRC, the validated method-side preprocessing policy uses a fixed global 4,000-HVG Seurat-v3 selection (`span=0.3`) because the donor/assay batch-aware LOESS variant was numerically unstable during validation. The same dataset policy is used across the Dataset 2 reference notebooks.

> **Validation note:** this notebook was tested on Google Colab runtime 2026.07. An optional, fully commented compatibility-check cell is included for diagnostics only. Exact NumPy/PyTorch/JAX version matching is **not required** to run the notebook.


## 1. Install the pinned scRareBench release

This notebook does not require a local source ZIP. It first bootstraps the pinned scRareBench release without dependencies, then calls the generic `scrarebench.runtime.setup_runtime()` helper. The method dependency is declared explicitly by this notebook; scRareBench itself does not know or register the method. The runtime helper checks dependency health, preserves the scientific ABI-sensitive packages already present in the current environment, and runs fresh-process import smoke tests. The separate compatibility-check cell is optional and disabled by default.


In [ ]:
SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.3"
METHOD = "harmony"

# To pin a release/commit, for development only, replace the release tag with the development branch explicitly.


In [ ]:
# OPTIONAL: validate this runtime against the environment used for release testing.
# This check is informational only and is NOT required to run scRareBench or this notebook.
# Leave this cell unchanged to skip the check. Uncomment the lines below if you want to compare
# your current environment with the Google Colab 2026.07 runtime used during validation.
# A different compatible runtime may still work correctly.
#
# import sys
# from importlib import metadata as _runtime_metadata
#
# _EXPECTED_COLAB_ANCHORS = {
#     "numpy": "2.0.2",
#     "torch": "2.11.0",
#     "jax": "0.7.2",
# }
# _runtime_mismatches = []
# for _package, _expected in _EXPECTED_COLAB_ANCHORS.items():
#     try:
#         _observed = _runtime_metadata.version(_package)
#     except _runtime_metadata.PackageNotFoundError:
#         _observed = "not installed"
#     if _observed != _expected:
#         _runtime_mismatches.append(
#             f"{_package}: validated {_expected}, current {_observed}"
#         )
#
# if _runtime_mismatches:
#     print("Runtime differs from the Colab 2026.07 validation environment:")
#     for _item in _runtime_mismatches:
#         print(" -", _item)
# else:
#     print("Runtime matches the documented Colab 2026.07 validation anchors.")


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

# Bootstrap only the lightweight scRareBench package code from the pinned release.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

from scrarebench.runtime import print_install_report, setup_runtime

# setup_runtime preserves ABI-sensitive scientific packages already present in the
# current environment. The optional compatibility-check cell above is informational
# only; matching the exact Colab 2026.07 anchor versions is not required.
install_report = setup_runtime(
    extra_requirements=('harmonypy==2.0.0',),
    extra_imports=('harmonypy',),
    quiet=False,
)
print_install_report(install_report)

# Import only after dependency validation and the fresh-process smoke test pass.
import scrarebench as _scrarebench_install_check
import scib_metrics as _scib_metrics_install_check
print("scrarebench import path:", Path(_scrarebench_install_check.__file__).resolve())
print("scrarebench version:", _scrarebench_install_check.__version__)
print("scib-metrics version:", getattr(_scib_metrics_install_check, "__version__", "installed"))
import harmonypy as _method_install_check
print("harmonypy version:", getattr(_method_install_check, "__version__", "2.0.0"))


## 2. Imports and environment information


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import hashlib
import importlib
import json
import os
import platform
import shutil
import sys
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import yaml


import scrarebench
from IPython.display import HTML, display

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("scRareBench:", scrarebench.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import harmonypy as hm
print("harmonypy:", getattr(hm,"__version__","installed"))


## 3. Harmony and Dataset 2 configuration


In [ ]:
WORK_DIR = Path("/content/scrarebench_harmony_dataset2_run")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = DATA_DIR / "cache"
RESULTS_DIR = WORK_DIR / "results" / "Harmony_dataset2"
ARTIFACT_DIR = WORK_DIR / "deliverable"
for directory in (WORK_DIR, DATA_DIR, CACHE_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATASET_SELECTOR = 2
DATASET_KEY = "mbdrc_renal_cortex"
LABEL_KEY = "cell_type"
BATCH_KEY = "scrarebench_batch"
COUNTS_LAYER = "counts"
LATENT_KEY = "X_pca_harmony"
METHOD_NAME = "Harmony"
SEED = 42
N_HVG = 4000

HVG_SELECTION_POLICY = "global_seurat_v3_raw_counts"
HVG_BATCH_KEY = None
HVG_SPAN = 0.3
SCIB_HVG_BATCH_MODE = "global"

N_NEIGHBORS = 15
REFERENCE_RESOLUTION = 1.0
RESOLUTION_SWEEP = (1.0,)
DISTANCE_METRIC = "euclidean"

RUN_SCIB = True
SCIB_N_HVG = 4000
SCIB_REFERENCE_N_PCS = 50
SCIB_N_JOBS = 1
SCIB_PROGRESS_BAR = True
SCIB_INCLUDE_SILHOUETTE_BATCH = True
SCIB_REQUIRE_SUCCESS = True

FORCE_REBUILD_DATASET = False
GENERATE_UMAP = True
GENERATE_INTERACTIVE_REPORT = True
GENERATE_PDF_REPORT = True
DOWNLOAD_RESULT_ZIP = True
DOWNLOAD_STANDALONE_REPORT = False
INCLUDE_LATENT_IN_BUNDLE = True

HTML_INCLUDE_OVERVIEW = True
HTML_INCLUDE_METRICS = True
HTML_INCLUDE_SCIB = True
HTML_INCLUDE_RARE = True
HTML_INCLUDE_RARE_UMAP = True
HTML_INCLUDE_RARE_HEATMAPS = True
HTML_INCLUDE_RARE_SCENARIO_ANALYSIS = True
HTML_INCLUDE_UMAP = True
HTML_INCLUDE_SANKEY = True
HTML_INCLUDE_REPRODUCIBILITY = True
HTML_INCLUDE_STATIC_FIGURES = True
HTML_INCLUDE_CELL_IDS = True

NORMALIZE_TARGET_SUM = 10_000.0
SCALE_MAX_VALUE = 10.0
N_PCS = 50
PCA_SOLVER = "arpack"
HARMONY_THETA = 2.0
HARMONY_LAMBDA = None
HARMONY_SIGMA = 0.1
HARMONY_N_CLUSTERS = None
HARMONY_TAU = 0.0
HARMONY_BLOCK_SIZE = 0.05
HARMONY_MAX_ITER = 10
HARMONY_MAX_ITER_KMEANS = 4
HARMONY_EPSILON_CLUSTER = 1e-3
HARMONY_EPSILON_HARMONY = 1e-2
HARMONY_ALPHA = 0.2
HARMONY_BATCH_PROP_CUTOFF = 1e-5
HARMONY_NCORES = 0
HARMONY_VERBOSE = True
REUSE_MATCHING_HARMONY_LATENT = True
HARMONY_CACHE_DIR = WORK_DIR / "harmony_cache"
HARMONY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)
print("Dataset selector:", DATASET_SELECTOR, DATASET_KEY)


## 4. Load Dataset 2, validate raw counts, and read registered scenario metadata


In [ ]:
def sampled_count_diagnostics(matrix, max_values: int = 100_000, seed: int = 42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    finite = values[np.isfinite(values)]
    return {
        "sampled_values": int(len(values)),
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "maximum": float(np.max(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }

def _copy_matrix(matrix):
    if sp.issparse(matrix):
        return matrix.copy().tocsr()
    return np.asarray(matrix).copy()

def ensure_counts_layer(adata, layer_key="counts"):
    """Create/validate a raw-count layer without changing cell order.

    Priority: existing layer -> CELLxGENE/raw slot aligned to current var_names -> X if X is count-like.
    """
    if layer_key in adata.layers:
        diag = sampled_count_diagnostics(adata.layers[layer_key], seed=SEED)
        if diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(f"Existing {layer_key!r} layer is not nonnegative integer-like raw counts: {diag}")

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)
        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = sampled_count_diagnostics(matrix, seed=SEED)
            if diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = _copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = sampled_count_diagnostics(adata.X, seed=SEED)
    if diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = _copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts. "
        f"layers={list(adata.layers)}, raw_present={adata.raw is not None}, X_diagnostics={diag}"
    )

def gex_mask(adata):
    for key in ("feature_types", "feature_type", "modality"):
        if key not in adata.var.columns:
            continue
        values = adata.var[key].astype(str).str.lower()
        mask = values.str.contains("gene expression|gex|rna", regex=True).to_numpy()
        if mask.any():
            return mask
    if "feature_biotype" in adata.var.columns:
        mask = adata.var["feature_biotype"].astype(str).str.lower().eq("gene").to_numpy()
        if mask.any():
            return mask
    return np.ones(adata.n_vars, dtype=bool)

def hash_strings(values) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\\0")
    return digest.hexdigest()


from scrarebench import dataset_info, load_dataset, resolve_dataset
from scrarebench.scenarios import scenario_table_from_adata

spec = resolve_dataset(DATASET_SELECTOR)
assert spec.key == DATASET_KEY, (spec.key, DATASET_KEY)
adata = load_dataset(
    DATASET_SELECTOR,
    DATA_DIR,
    force_download=False,
    force_rebuild=FORCE_REBUILD_DATASET,
    strict_expected_counts=True,
)
if not adata.obs_names.is_unique:
    raise ValueError("Cell barcodes / obs_names are not unique.")

# Dataset 2: mBDRC renal cortex. load_dataset(2) attaches the registered
# evaluation batch (donor_id x assay) and the fixed registered rare-scenario taxonomy.
info = dataset_info(adata)
if info.get("label_key") != LABEL_KEY or info.get("batch_key") != BATCH_KEY:
    raise RuntimeError(f"Unexpected Dataset 2 evaluation metadata: {info}")
for required_key in ("donor_id", "assay", LABEL_KEY, BATCH_KEY):
    if required_key not in adata.obs.columns:
        raise KeyError(
            f"Dataset 2 requires obs[{required_key!r}]. Available columns: {list(adata.obs.columns)}"
        )

count_source, count_diagnostics = ensure_counts_layer(adata, COUNTS_LAYER)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError("Dataset 2 registered rare-scenario metadata was not attached by load_dataset(2).")
scenario_registry = adata.uns.get("scrarebench_scenario_registry", {})

print(adata)
print("Cells:", f"{adata.n_obs:,}")
print("Features:", f"{adata.n_vars:,}")
print("Batches:", adata.obs[BATCH_KEY].astype(str).nunique())
print("Cell types:", adata.obs[LABEL_KEY].astype(str).nunique())
print("Registered rare-scenario coverage:", scenario_registry.get("scenario_coverage", []))
print("Registered six-state rare populations:", len(scenario_table))
display(scenario_table[["cell_type", "scenario", "distribution", "topology"]])
print("Rare populations used by scRareBench:")
display(scenario_table)


## 5. HVG selection, normalization, PCA, and Harmony-specific preprocessing


In [ ]:
# Fixed, dataset-level HVG policy. It is identical across methods on the same dataset.
# Dataset 2 audit: global Seurat-v3 HVG at span=0.3 passed, while donor_id+assay batch-aware HVG remained numerically singular. The same fixed global HVG policy is used for scVI, Harmony and MrVI on Dataset 2.
rna_mask = gex_mask(adata)
adata_rna = adata[:, rna_mask].copy() if int(rna_mask.sum()) != adata.n_vars else adata.copy()
adata_rna.var_names_make_unique()

if COUNTS_LAYER not in adata_rna.layers:
    raise KeyError(f"{COUNTS_LAYER!r} layer is missing after count preparation.")

count_diagnostics = sampled_count_diagnostics(adata_rna.layers[COUNTS_LAYER], seed=SEED)
if not count_diagnostics["nonnegative"] or not count_diagnostics["integer_like"]:
    raise ValueError(f"Selected count matrix is invalid: {count_diagnostics}")

hvg_kwargs = {
    "layer": COUNTS_LAYER,
    "flavor": "seurat_v3",
    "n_top_genes": min(N_HVG, adata_rna.n_vars),
    "span": HVG_SPAN,
    "subset": False,
    "check_values": True,
}
if HVG_BATCH_KEY is not None:
    if HVG_BATCH_KEY not in adata_rna.obs.columns:
        raise KeyError(
            f"HVG_BATCH_KEY={HVG_BATCH_KEY!r} is missing from adata.obs. "
            f"Available={list(adata_rna.obs.columns)}"
        )
    hvg_kwargs["batch_key"] = HVG_BATCH_KEY

print("HVG selection policy:", HVG_SELECTION_POLICY)
print("HVG batch key:", HVG_BATCH_KEY if HVG_BATCH_KEY is not None else "<global / no batch_key>")
print("HVG span:", HVG_SPAN)

sc.pp.highly_variable_genes(adata_rna, **hvg_kwargs)

hvg_mask = adata_rna.var["highly_variable"].fillna(False).to_numpy()
n_selected_hvg = int(hvg_mask.sum())
if n_selected_hvg == 0:
    raise RuntimeError("No HVGs were selected.")
if n_selected_hvg != min(N_HVG, adata_rna.n_vars):
    raise RuntimeError(
        f"Expected {min(N_HVG, adata_rna.n_vars)} HVGs but selected {n_selected_hvg}. "
        "Stop here rather than silently changing preprocessing across methods."
    )

hvg_names = adata_rna.var_names[hvg_mask]
counts_hvg = adata_rna.layers[COUNTS_LAYER][:, hvg_mask]
if sp.issparse(counts_hvg):
    counts_hvg = counts_hvg.tocsr().astype(np.float32)
else:
    counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

adata_harmony = ad.AnnData(
    X=counts_hvg.copy(),
    obs=adata_rna.obs.copy(),
    var=adata_rna.var.loc[hvg_names].copy(),
)
adata_harmony.var_names_make_unique()

if not np.array_equal(adata_harmony.obs_names.astype(str), adata.obs_names.astype(str)):
    raise RuntimeError("Cell order changed during method preprocessing.")

print(adata_harmony)
print("Selected HVGs:", adata_harmony.n_vars)
print("HVG policy hash input:", HVG_SELECTION_POLICY, HVG_BATCH_KEY, HVG_SPAN)

# Harmony-specific expression preprocessing after the common fixed HVG selection.
sc.pp.normalize_total(adata_harmony, target_sum=NORMALIZE_TARGET_SUM)
sc.pp.log1p(adata_harmony)
sc.pp.scale(adata_harmony, max_value=SCALE_MAX_VALUE)
sc.tl.pca(
    adata_harmony,
    n_comps=min(N_PCS, adata_harmony.n_vars - 1, adata_harmony.n_obs - 1),
    svd_solver=PCA_SOLVER,
    random_state=SEED,
)
print("PCA shape:", adata_harmony.obsm["X_pca"].shape)


## 6. Run Harmony


In [ ]:
harmony_config = {
    "seed": SEED, "n_hvg": N_HVG, "n_pcs": int(adata_harmony.obsm["X_pca"].shape[1]),
    "theta": HARMONY_THETA, "lamb": HARMONY_LAMBDA, "sigma": HARMONY_SIGMA,
    "nclust": HARMONY_N_CLUSTERS, "tau": HARMONY_TAU, "block_size": HARMONY_BLOCK_SIZE,
    "max_iter_harmony": HARMONY_MAX_ITER, "max_iter_kmeans": HARMONY_MAX_ITER_KMEANS,
    "epsilon_cluster": HARMONY_EPSILON_CLUSTER, "epsilon_harmony": HARMONY_EPSILON_HARMONY,
    "alpha": HARMONY_ALPHA, "batch_prop_cutoff": HARMONY_BATCH_PROP_CUTOFF,
    "ncores": HARMONY_NCORES, "batch_key": BATCH_KEY, "dataset_selector": DATASET_SELECTOR,
    "hvg_selection_policy": HVG_SELECTION_POLICY,
    "hvg_batch_key": HVG_BATCH_KEY,
    "hvg_span": HVG_SPAN,
}
manifest = {
    "cell_hash": hash_strings(adata_harmony.obs_names),
    "gene_hash": hash_strings(adata_harmony.var_names),
    "batch_hash": hash_strings(adata_harmony.obs[BATCH_KEY].astype(str)),
    "config": harmony_config,
}
manifest_path=HARMONY_CACHE_DIR/"run_manifest.json"; latent_cache=HARMONY_CACHE_DIR/"X_pca_harmony.npy"; objective_cache=HARMONY_CACHE_DIR/"objectives.json"
can_reuse=False
if REUSE_MATCHING_HARMONY_LATENT and manifest_path.exists() and latent_cache.exists():
    try: can_reuse=json.loads(manifest_path.read_text())==manifest
    except Exception: can_reuse=False
if can_reuse:
    X_latent=np.load(latent_cache).astype(np.float32); training_seconds=0.0
    harmony_objectives=json.loads(objective_cache.read_text()) if objective_cache.exists() else {}
    print("Loaded cached Harmony latent.")
else:
    start=time.perf_counter()
    ho=hm.run_harmony(
        adata_harmony.obsm["X_pca"], adata_harmony.obs, BATCH_KEY,
        theta=HARMONY_THETA, lamb=HARMONY_LAMBDA, sigma=HARMONY_SIGMA,
        nclust=HARMONY_N_CLUSTERS, tau=HARMONY_TAU, block_size=HARMONY_BLOCK_SIZE,
        max_iter_harmony=HARMONY_MAX_ITER, max_iter_kmeans=HARMONY_MAX_ITER_KMEANS,
        epsilon_cluster=HARMONY_EPSILON_CLUSTER, epsilon_harmony=HARMONY_EPSILON_HARMONY,
        alpha=HARMONY_ALPHA, batch_prop_cutoff=HARMONY_BATCH_PROP_CUTOFF,
        ncores=HARMONY_NCORES, random_state=SEED, verbose=HARMONY_VERBOSE,
    )
    training_seconds=time.perf_counter()-start
    X_latent=np.asarray(ho.Z_corr.T, dtype=np.float32)
    harmony_objectives={
        "objective_harmony": np.asarray(getattr(ho,"objective_harmony",[]),dtype=float).tolist(),
        "objective_kmeans": np.asarray(getattr(ho,"objective_kmeans",[]),dtype=float).tolist(),
    }
    np.save(latent_cache,X_latent); manifest_path.write_text(json.dumps(manifest,indent=2)); objective_cache.write_text(json.dumps(harmony_objectives,indent=2))
print("Harmony latent:", X_latent.shape)
print(f"Harmony time: {training_seconds/60:.2f} min")


## 7. Attach the latent and validate cell alignment


In [ ]:
from scrarebench.latent import attach_latent
barcodes=adata_harmony.obs_names.astype(str).to_numpy()
if X_latent.shape[0] != adata.n_obs: raise RuntimeError(f"Harmony latent row mismatch: {X_latent.shape}")
if not np.isfinite(X_latent).all(): raise ValueError("Harmony latent contains NaN/Inf")
alignment_report=attach_latent(adata,X_latent,key=LATENT_KEY,latent_barcodes=barcodes,allow_reorder=False,overwrite=True)
np.save(WORK_DIR/"Harmony_dataset2_latent.npy",X_latent); np.save(WORK_DIR/"Harmony_dataset2_barcodes.npy",barcodes)
print("Alignment:",alignment_report)
del adata_harmony, counts_hvg
gc.collect()


## 8. Run the scRareBench evaluation


In [ ]:
from scrarebench.evaluation import EvaluationConfig, evaluate_latent
from scrarebench.scib_backend import ScibEvaluationConfig

if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_config = EvaluationConfig(
    method_name=METHOD_NAME,
    representation_key=LATENT_KEY,
    label_key=LABEL_KEY,
    batch_key=BATCH_KEY,
    scenario_key="scrarebench_scenario",
    reference_resolution=REFERENCE_RESOLUTION,
    resolution_sweep=RESOLUTION_SWEEP,
    n_neighbors=N_NEIGHBORS,
    distance_metric=DISTANCE_METRIC,
    random_state=SEED,
    overwrite=True,
    scib=ScibEvaluationConfig(
        enabled=RUN_SCIB,
        count_layer=COUNTS_LAYER,
        n_hvg=SCIB_N_HVG,
        hvg_batch_mode=SCIB_HVG_BATCH_MODE,
        reference_n_pcs=SCIB_REFERENCE_N_PCS,
        n_jobs=SCIB_N_JOBS,
        progress_bar=SCIB_PROGRESS_BAR,
        include_silhouette_batch=SCIB_INCLUDE_SILHOUETTE_BATCH,
        require_backend=SCIB_REQUIRE_SUCCESS,
    ),
)

# Use the Dataset 2 registered scenario table attached by load_dataset(2).
result = evaluate_latent(
    adata,
    benchmark_config,
    RESULTS_DIR,
    scenario_table=scenario_table,
)

print("Benchmark completed.")
print("Reference cluster key:", result.cluster_keys[REFERENCE_RESOLUTION])
print("\\nOverall / rare / non-rare metrics")
display(result.subset_metrics.round(5))
print("\\nRare-cell summary")
display(result.rare_summary.round(5))
print("\\nRare-cell per-type metrics")
display(result.rare_metrics.round(5))
if result.scib is not None:
    print("\\nscIB-compatible aggregate scores")
    display(result.scib.aggregate_scores.round(5))
    print("\\nscIB-compatible individual metrics")
    display(result.scib.metrics_long.round(5))
    print("\\nMetric applicability/status")
    display(result.scib.metric_status)


## 9. Build UMAP and report artifacts


In [ ]:
from scrarebench.reporting import write_html_report, write_interactive_report, write_pdf_report

figures_dir = RESULTS_DIR / "rare_cell" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
extra_figures = []
run_config = yaml.safe_load((RESULTS_DIR / "reproducibility" / "run_config.yaml").read_text(encoding="utf-8"))
neighbors_key = run_config["neighbors_key"]
reference_cluster_key = run_config["reference_cluster_key"]

UMAP_KEY = "X_umap_Harmony_dataset2"
if GENERATE_UMAP:
    sc.tl.umap(adata, neighbors_key=neighbors_key, random_state=SEED)
    adata.obsm[UMAP_KEY] = adata.obsm["X_umap"].copy()
    for color_key, title, filename, size in [
        (LABEL_KEY, "Harmony latent UMAP — reference cell types", "umap_cell_types.png", (13, 9)),
        (BATCH_KEY, "Harmony latent UMAP — benchmark batches", "umap_batches.png", (12, 8)),
    ]:
        sc.pl.embedding(
            adata,
            basis=UMAP_KEY,
            color=color_key,
            title=title,
            frameon=False,
            show=False,
            legend_loc="right margin",
        )
        path = figures_dir / filename
        plt.gcf().set_size_inches(*size)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

    rare_mask = adata.obs["scrarebench_is_rare"].astype(bool).to_numpy()
    if rare_mask.any():
        rare_view = adata[rare_mask].copy()
        sc.pl.embedding(
            rare_view,
            basis=UMAP_KEY,
            color="scrarebench_scenario",
            title="Registered mBDRC rare scenarios",
            frameon=False,
            show=False,
            legend_loc="right margin",
            size=24,
        )
        path = figures_dir / "umap_rare_distribution_classes.png"
        plt.gcf().set_size_inches(11, 8)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

obj=np.asarray(harmony_objectives.get("objective_harmony",[]),dtype=float)
if obj.size:
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot(np.arange(len(obj)),obj,marker="o",ms=3); ax.set_xlabel("Harmony iteration"); ax.set_ylabel("Objective"); ax.set_title("Harmony objective"); fig.tight_layout()
    p=figures_dir/"harmony_objective.png"; fig.savefig(p,dpi=180,bbox_inches="tight"); plt.close(fig); extra_figures.append(p)


base_figures = []
if result.scib is not None:
    base_figures.append(result.scib.files["metric_plot"])
base_figures.extend([
    result.files["rare_metric_heatmap"],
    result.files["rare_precision_recall"],
    result.files["failure_counts"],
])

report_metadata = {
    "method": METHOD_NAME,
    "dataset_selector": DATASET_SELECTOR,
    "dataset_key": DATASET_KEY,
    "representation_key": LATENT_KEY,
    "n_cells": adata.n_obs,
    "n_dimensions": adata.obsm[LATENT_KEY].shape[1],
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "rare_definition": "registered mBDRC six-state scenario taxonomy",
    "reference_resolution": REFERENCE_RESOLUTION,
    "n_neighbors": N_NEIGHBORS,
    "distance_metric": DISTANCE_METRIC,
    "benchmark_seed": SEED,
    "reference_cluster_key": reference_cluster_key,
    "cell_order_exact_match": alignment_report["exact_order_match"],
    "scib_backend": result.scib.backend if result.scib else "disabled",
    "scib_backend_version": result.scib.backend_version if result.scib else "n/a",
    "Harmony_n_hvg": N_HVG, "Harmony_n_pcs": N_PCS, "Harmony_seconds": round(training_seconds,2),
}

report_path = RESULTS_DIR / "report.html"
write_html_report(
    report_path,
    title="scRareBench report — Harmony on Dataset 2: mBDRC renal cortex",
    metadata=report_metadata,
    global_table=result.subset_metrics,
    rare_table=result.rare_metrics,
    figure_names=base_figures + extra_figures,
    scib_metrics=result.scib.metrics_long if result.scib else None,
    scib_aggregates=result.scib.aggregate_scores if result.scib else None,
    scib_status=result.scib.metric_status if result.scib else None,
    rare_summary=result.rare_summary,
    scenario_table=result.scenario_metrics,
)

for index, figure_path in enumerate(extra_figures):
    result.files[f"notebook_figure_{index:02d}"] = figure_path

interactive_report_path = RESULTS_DIR / "interactive_report.html"
pdf_report_path = RESULTS_DIR / "summary_report.pdf"
HTML_REPORT_OPTIONS = {
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "scenario_key": "scrarebench_scenario",
    "umap_key": UMAP_KEY if UMAP_KEY in adata.obsm else None,
    "include_overview": HTML_INCLUDE_OVERVIEW,
    "include_metrics": HTML_INCLUDE_METRICS,
    "include_scib": HTML_INCLUDE_SCIB,
    "include_rare": HTML_INCLUDE_RARE,
    "include_rare_umap": HTML_INCLUDE_RARE_UMAP,
    "include_rare_heatmaps": HTML_INCLUDE_RARE_HEATMAPS,
    "include_rare_scenario_analysis": HTML_INCLUDE_RARE_SCENARIO_ANALYSIS,
    "include_umap": HTML_INCLUDE_UMAP,
    "include_sankey": HTML_INCLUDE_SANKEY,
    "include_reproducibility": HTML_INCLUDE_REPRODUCIBILITY,
    "include_static_figures": HTML_INCLUDE_STATIC_FIGURES,
    "include_cell_ids": HTML_INCLUDE_CELL_IDS,
}
if GENERATE_INTERACTIVE_REPORT:
    write_interactive_report(
        adata,
        result,
        interactive_report_path,
        representation_key=LATENT_KEY,
        **HTML_REPORT_OPTIONS,
    )
if GENERATE_PDF_REPORT:
    write_pdf_report(adata, result, pdf_report_path, representation_key=LATENT_KEY)

print("Static HTML report:", report_path)
print("Interactive HTML report:", interactive_report_path if GENERATE_INTERACTIVE_REPORT else "disabled")
print("PDF report:", pdf_report_path if GENERATE_PDF_REPORT else "disabled")


## 10. Display and inspect generated reports


In [ ]:
display(HTML(report_path.read_text(encoding="utf-8")))

print("\\nGenerated output files:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULTS_DIR))


## 11. Build and download the result bundle


In [ ]:
from scrarebench.reporting import create_report_bundle

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "method": METHOD_NAME,
    "dataset_selector": DATASET_SELECTOR,
    "dataset": "Dataset 2: mBDRC renal cortex",
    "n_cells": int(adata.n_obs),
    "latent_shape": list(adata.obsm[LATENT_KEY].shape),
    "rare_definition": "distribution-only GR/LE/SR; topology UNASSIGNED",
    "include_latent": INCLUDE_LATENT_IN_BUNDLE,
}
(ARTIFACT_DIR / "README_OUTPUT.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

archive_path = create_report_bundle(
    adata,
    result,
    WORK_DIR / "Harmony_dataset2_mBDRC_scRareBench.zip",
    representation_key=LATENT_KEY,
    include_latent=INCLUDE_LATENT_IN_BUNDLE,
    write_interactive=GENERATE_INTERACTIVE_REPORT,
    write_pdf=GENERATE_PDF_REPORT,
    interactive_report_options=HTML_REPORT_OPTIONS,
)
print("Final ZIP:", archive_path)
print("ZIP size (MB):", round(archive_path.stat().st_size / 1024 / 1024, 2))

if DOWNLOAD_RESULT_ZIP:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception as exc:
        print("Automatic Colab download was not available:", exc)


## Interpretation notes

- scRareBench does not run Harmony; it evaluates the latent produced by this notebook.
- Dataset 2 is evaluated with the registered composite batch `donor_id x assay`.
- Rare-cell metrics use the fixed registered mBDRC scenario taxonomy attached by `load_dataset(2)`; the notebook does not infer a competing taxonomy.
- `HVG_SELECTION_POLICY`, `HVG_BATCH_KEY`, `HVG_SPAN`, and `SCIB_HVG_BATCH_MODE` are explicit so preprocessing/evaluation policy is reproducible across methods on the same dataset.
- No adaptive per-method fallback is used: if the fixed method-side preprocessing policy fails, the notebook stops rather than silently changing the benchmark recipe.
